# CareRisk 48H — clean Colab training

Use `STAGE='prepare'` on a CPU runtime, then `STAGE='train'` on L4. Quick mode is synthetic smoke only; full mode uses only PhysioNet Set A. Source is cloned from a checksummed immutable Git bundle, while data, checkpoints, artifacts, and result packages persist on Drive. No cell downloads or accesses Set B or Set C.

In [ ]:
STAGE = 'prepare'  # 'prepare' on CPU, then 'train' on L4
MODE = 'quick'  # during train stage: 'quick' first, then 'full'
EXPECTED_GPU = 'L4'  # change to 'A100' only after recorded OOM/runtime/speed evidence
HANDOFF_DIR = '/content/drive/MyDrive/CareRisk48H-handoff'
PERSISTENT_DIR = '/content/drive/MyDrive/CareRisk48H-runtime'
RESUME = True
assert STAGE in {'prepare', 'train'}
assert MODE in {'quick', 'full'}
assert EXPECTED_GPU in {'L4', 'A100'}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

handoff = Path(HANDOFF_DIR)
receipt = json.loads((handoff / 'carerisk48h-source-receipt.json').read_text(encoding='utf-8'))
bundle = handoff / receipt['bundle_filename']
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
assert bundle.is_file()
assert file_sha256(bundle) == receipt['bundle_sha256']
project = Path('/content/CareRisk48H-source')
if project.exists():
    assert project == Path('/content/CareRisk48H-source')
    shutil.rmtree(project)
subprocess.run(['git', 'clone', '--branch', receipt['source_branch'], str(bundle), str(project)], check=True)
source_git_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=project, check=True, capture_output=True, text=True).stdout.strip()
assert source_git_sha == receipt['source_git_sha']
assert not subprocess.run(['git', 'status', '--porcelain'], cwd=project, check=True, capture_output=True, text=True).stdout.strip()
persistent = Path(PERSISTENT_DIR)
persistent.mkdir(parents=True, exist_ok=True)
for name in ('data', 'artifacts', 'checkpoints'):
    target = persistent / name
    target.mkdir(parents=True, exist_ok=True)
    link = project / name
    if link.exists() or link.is_symlink():
        raise FileExistsError(f'Clean source unexpectedly contains {link}')
    link.symlink_to(target, target_is_directory=True)
os.chdir(project)
print(f'Verified immutable source: {source_git_sha}')
print(f'Persistent runtime state: {persistent}')

In [ ]:
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps
import importlib.metadata as md, platform, torch
assert (3, 10) <= sys.version_info[:2] < (3, 13)
assert int(torch.__version__.split('+')[0].split('.')[0]) >= 2
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
versions = {name: md.version(name) for name in ['carerisk48h', 'numpy', 'pandas', 'scikit-learn', 'torch', 'lightgbm', 'shap']}
runtime = {
    'python': sys.version,
    'platform': platform.platform(),
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'packages': versions,
}
print(json.dumps(runtime, indent=2))
environment_lock = project / 'artifacts' / f'environment-{MODE}.lock.txt'
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True).stdout
environment_lock.write_text(json.dumps(runtime, sort_keys=True) + '\n# pip freeze\n' + freeze, encoding='utf-8')
print(f'Environment lock SHA-256: {file_sha256(environment_lock)}')

In [ ]:
if STAGE == 'prepare':
    if torch.cuda.is_available():
        raise RuntimeError('Switch Colab to a CPU runtime for Set A download and preparation.')
    manifest = project / 'data/raw/manifest-set-a.json'
    if not manifest.is_file():
        subprocess.run([sys.executable, 'scripts/download_physionet.py', '--raw-dir', 'data/raw', '--set', 'a'], check=True)
    subprocess.run([sys.executable, 'scripts/download_physionet.py', '--raw-dir', 'data/raw', '--verify-only', str(manifest)], check=True)
    subprocess.run([sys.executable, 'scripts/generate_data_quality.py'], check=True)
    split_path = project / 'data/processed/set_a_split.csv'
    import pandas as pd
    from carerisk48h.artifacts import stable_hash
    split = pd.read_csv(split_path)
    assert split['split'].value_counts().to_dict() == {'train': 2800, 'validation': 600, 'calibration': 600}
    semantic_split_hash = stable_hash(split.to_dict(orient='records'))
    assert semantic_split_hash == '77a2f0ddccbe5d27f4a580c0918499435105b67e944908fa295bfb43d7ffd631'
    print(f'Set A manifest SHA-256: {file_sha256(manifest)}')
    print(f'Frozen semantic split hash: {semantic_split_hash}')
    print('CPU preparation complete. Change STAGE to train and switch to L4.')
else:
    print('Preparation skipped in train stage; no downloader command will run.')

In [ ]:
completed_runs = []
if STAGE == 'train':
    if not torch.cuda.is_available():
        raise RuntimeError('Training stage requires the requested L4 GPU runtime.')
    gpu_name = torch.cuda.get_device_name(0)
    if EXPECTED_GPU.lower() not in gpu_name.lower():
        raise RuntimeError(f'Selected {gpu_name}; follow the requested L4-first policy (expected {EXPECTED_GPU}).')
    if MODE == 'full':
        required_set_a = [project / 'data/raw/set-a', project / 'data/raw/manifest-set-a.json', project / 'data/processed/set_a_split.csv']
        if any(not path.exists() for path in required_set_a):
            raise FileNotFoundError('Full mode needs prepared Set A and its frozen split on Drive.')
    config = f'configs/{MODE}.yaml'
    checkpoint_dir = project / 'checkpoints' / MODE
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    common = ['--config', config, '--device', 'cuda', '--checkpoint-dir', str(checkpoint_dir)]
    if MODE == 'quick':
        common += ['--synthetic']
    if RESUME:
        common += ['--resume']
    for family in ('grud', 'tcn'):
        before = set((project / 'artifacts').glob(f'*-{family}-{MODE}'))
        subprocess.run([sys.executable, 'scripts/train_deep.py', '--family', family, *common], check=True)
        created = set((project / 'artifacts').glob(f'*-{family}-{MODE}')) - before
        if len(created) != 1:
            raise RuntimeError(f'Expected one new {family} run, found {len(created)}')
        completed_runs.append(created.pop())
    print('Completed runs:', [path.name for path in completed_runs])
else:
    print('Training skipped during CPU preparation stage.')

In [ ]:
if STAGE == 'train':
    from carerisk48h.colab_handoff import package_deep_results
    package, checksum = package_deep_results(
        completed_runs,
        checkpoint_dir=project / 'checkpoints' / MODE,
        environment_lock=environment_lock,
        output_dir=project / 'artifacts' / 'colab-results',
        mode=MODE,
        expected_git_sha=source_git_sha,
    )
    print(f'Result package: {package}')
    print(f'Package bytes: {package.stat().st_size}')
    print(f'Package SHA-256: {file_sha256(package)}')
    print(f'Checksum sidecar: {checksum}')
    print('Quick packages are smoke_test evidence only and can never update formal results.')
else:
    print('No result package is produced during preparation.')

## Runtime sequence

1. CPU clean runtime: `STAGE='prepare'`, Run all.
2. L4 clean runtime: `STAGE='train'`, `MODE='quick'`, Run all.
3. If quick passes, L4 clean runtime: `STAGE='train'`, `MODE='full'`, Run all.
4. Return the full ZIP and `.sha256` from `CareRisk48H-runtime/artifacts/colab-results/`. Calibration, refit, freeze, and any final-test access remain separate gated steps.